In [55]:
import numpy as np
import pandas as pd

from collections import defaultdict
from Bio import SeqIO


In [56]:
import numpy as np

def filter_outliers(data, k = 1.5, return_idx = False, method = 'iqr'):
    """Remove outliers outside [Q1 - k*IQR, Q3 + k*IQR]."""
    data = np.asarray(data)
    q1, q3 = np.percentile(data, [25, 75])
    iqr = q3 - q1
    lower, upper = q1 - k * iqr, q3 + k * iqr
    mask = (data >= lower) & (data <= upper)
    if return_idx:
        return np.where(mask)[0], data[mask]
    else:
        return data[mask]

Preprocess CAR entries from InterPro DB
----------------------------------------

In [81]:
fasta_path = 'car_interpro.fasta'

seq_to_records = defaultdict(list)

for record in SeqIO.parse(fasta_path, "fasta"):
    desc = record.description.split('|')
    uniprot_id = desc[0]
    is_reviewed = desc[1]
    function = desc[2]
    tax_id = desc[3]
    seq = str(record.seq)  # cast to str so it is hashable as a dict key
    
    seq_to_records[seq].append((uniprot_id, is_reviewed, function, tax_id))

# Build DataFrame: one row per unique sequence
rows = []
for seq, members in seq_to_records.items():
    uniprot_ids = [m[0] for m in members]
    
    reviewed_ids = [uid for uid, status, _, _ in members if status == 'reviewed']
    is_reviewed = f"reviewed ({','.join(reviewed_ids)})" if reviewed_ids else 'unreviewed'
    
    rows.append({
        'uniprot_id': ';'.join(uniprot_ids),
        'is_reviewed': is_reviewed,
        'function': ';'.join(set(m[2] for m in members)),
        'tax_id': ';'.join(m[3] for m in members),
        'length': len(seq),
        'sequence': seq,
    })

df_interpro = pd.DataFrame(rows)

print(f'Redundace: {len(list(SeqIO.parse(fasta_path, "fasta")))}, None-redundace: {len(df_interpro)}')
df_interpro

Redundace: 1026, None-redundace: 986


,uniprot_id,is_reviewed,function,tax_id,length,sequence
0,A0A010YLP6,unreviewed,Carrier domain-containing protein,taxID:927661,1152,MTHTDQGDVTTVAERRALLLEQDEQFASAVPDDHVTALVRQPTMSL...
1,A0A024JSK2,unreviewed,Carboxylic acid reductase,taxID:47839,1188,MTTDMKRENGTNSTTKATPRERVAERIRRVEATDEQYRNAKPDQAL...
2,A0A024JWI0,unreviewed,Carboxylic acid reductase,taxID:47839,1162,MSTTTREERLERRIENLTATDPQFAAAKPDPAVVEALEQPGLQLPQ...
3,A0A024JZE3,unreviewed,Carboxylic acid reductase,taxID:47839,1165,MAFVGRSDVKDPGAQQDQWERLARRRERLYAEDAQFAATRPDEQIA...
4,A0A024JZE8,unreviewed,Carboxylic acid reductase,taxID:47839,1182,MTSGSLRDIHLAEPGDNRDERTAQRVAELFDNDPQFRAAAPLPDVI...
...,...,...,...,...,...,...
981,X8CEN6,unreviewed,Carboxylic acid reductase,taxID:1299331,1162,MAATDEQFRNAQPDLSLQQAARQPGLRLPQILELFVEGYADRPAVG...
982,X8CIS2,unreviewed,Carboxylic acid reductase,taxID:1299331,1072,MSTAIHDENLDRRIEELIANDPQFAAARPDPAITAATEAPGLRLPQ...
983,X8DPU6,unreviewed,Carboxylic acid reductase,taxID:1299321,1183,MTIDATADNTKEARRQRLGDRVRRLFTEDEQFRAAKPDTAVDAAVA...
984,X8DT62,unreviewed,Carrier domain-containing protein,taxID:1299321,902,MAGTENLPMIGLNFMPMSHIMGRGTLTSTLSTGGTGYFAASSDMST...


In [85]:
idx, values = filter_outliers(df_interpro['length'], return_idx = True)
df_filtered = df_interpro.iloc[idx]
df_filtered

,uniprot_id,is_reviewed,function,tax_id,length,sequence
0,A0A010YLP6,unreviewed,Carrier domain-containing protein,taxID:927661,1152,MTHTDQGDVTTVAERRALLLEQDEQFASAVPDDHVTALVRQPTMSL...
1,A0A024JSK2,unreviewed,Carboxylic acid reductase,taxID:47839,1188,MTTDMKRENGTNSTTKATPRERVAERIRRVEATDEQYRNAKPDQAL...
2,A0A024JWI0,unreviewed,Carboxylic acid reductase,taxID:47839,1162,MSTTTREERLERRIENLTATDPQFAAAKPDPAVVEALEQPGLQLPQ...
3,A0A024JZE3,unreviewed,Carboxylic acid reductase,taxID:47839,1165,MAFVGRSDVKDPGAQQDQWERLARRRERLYAEDAQFAATRPDEQIA...
4,A0A024JZE8,unreviewed,Carboxylic acid reductase,taxID:47839,1182,MTSGSLRDIHLAEPGDNRDERTAQRVAELFDNDPQFRAAAPLPDVI...
...,...,...,...,...,...,...
979,X0Q040,unreviewed,Carboxylic acid reductase,taxID:1219028,1175,MSTDIREERLARRIADLYANDRQFVAARPSEALTAAIEQPGLRLPQ...
980,X8A9W7;X8CDI5,unreviewed,Carboxylic acid reductase,taxID:1767;taxID:1299331,1188,MTDTVTDSGREQRLTERVEQLYANDPQFRAAAPSPEVTEAAHRAGL...
981,X8CEN6,unreviewed,Carboxylic acid reductase,taxID:1299331,1162,MAATDEQFRNAQPDLSLQQAARQPGLRLPQILELFVEGYADRPAVG...
983,X8DPU6,unreviewed,Carboxylic acid reductase,taxID:1299321,1183,MTIDATADNTKEARRQRLGDRVRRLFTEDEQFRAAKPDTAVDAAVA...


Load the PDB files of the CAR homologs by parsing the UniProt
--

In [88]:
import requests
import pandas as pd
import os
import time

from tqdm.notebook import tqdm
# ── Load data ──────────────────────────────────────────────────────────────────
# `df` is the deduplicated DataFrame from the previous step
# (uniprot_id column may hold comma-separated IDs that share the same sequence)
print(f"Total unique sequences: {len(df)}")
os.makedirs("structures", exist_ok=True)

def get_alphafold_pdb_url(uniprot_id: str) -> str:
    """
    Query AlphaFold EBI API to get the actual PDB download URL.
    The API returns metadata including 'pdbUrl' which points to the exact file.
    This avoids hardcoding version numbers (v1/v2/v3/v4).
    
    API endpoint: https://alphafold.ebi.ac.uk/api/prediction/{uniprot_id}
    """
    url = f"https://alphafold.ebi.ac.uk/api/prediction/{uniprot_id}"
    res = requests.get(url, timeout=10)
    if res.status_code != 200:
        return None
    data = res.json()
    if not data:
        return None
    return data[0].get("pdbUrl")

def download_structure(pdb_url: str, save_path: str) -> bool:
    """Download PDB file from a direct URL and save to disk."""
    res = requests.get(pdb_url, timeout=30)
    if res.status_code == 200:
        with open(save_path, "w") as f:
            f.write(res.text)
        return True
    return False

# ── Main loop ──────────────────────────────────────────────────────────────────
for ids_str in tqdm(df['uniprot_id']):
    uniprot_ids = ids_str.split(';')
    
    # Skip if any ID in this group already has a downloaded structure
    if any(os.path.exists(f"structures/car_{uid}.pdb") for uid in uniprot_ids):
        print(f"[SKIP]  {ids_str}")
        continue
    
    # Try each uniprot_id in order; stop at the first one that yields a structure
    success_uid = None
    for uid in uniprot_ids:
        pdb_url = get_alphafold_pdb_url(uid)
        if pdb_url and download_structure(pdb_url, f"structures/car_{uid}.pdb"):
            success_uid = uid
            break
        time.sleep(0.3)  # rate-limit between fallback attempts within a group
    
    if success_uid:
        tag = f"{success_uid}" + (f"  (from group of {len(uniprot_ids)})" if len(uniprot_ids) > 1 else "")
        print(f"[SUCCESS]    {tag}")
    else:
        print(f"[FAIL]  {ids_str} — no structure found for any ID")
    
    time.sleep(0.3)

Total unique sequences: 986


  0%|          | 0/986 [00:00<?, ?it/s]

[SKIP]  A0A010YLP6
[SKIP]  A0A024JSK2
[SKIP]  A0A024JWI0
[SKIP]  A0A024JZE3
[SKIP]  A0A024JZE8
[SUCCESS]    A0A045K2P5  (from group of 8)
[SKIP]  A0A051U3G5
[SKIP]  A0A051UFG9
[SKIP]  A0A064CG00
[SKIP]  A0A0B1ZH49
[SKIP]  A0A0B1ZIQ0
[SKIP]  A0A0B8NM02
[SKIP]  A0A0D1L7Y1
[SUCCESS]    A0A0D1LMF4
[SUCCESS]    A0A0D2EWW5
[SUCCESS]    A0A0E4CLA0
[SUCCESS]    A0A0E4GX90
[SUCCESS]    A0A0F4ES51
[SUCCESS]    A0A0F5MSZ2
[SUCCESS]    A0A0F5NA60
[SUCCESS]    A0A0H2ZUI5
[SUCCESS]    A0A0H3MCY6  (from group of 2)
[SUCCESS]    A0A0H3MU55  (from group of 3)
[SUCCESS]    A0A0H5BB76
[SUCCESS]    A0A0H5NNV9
[SUCCESS]    A0A0H5RL48
[SUCCESS]    A0A0H5RMP7
[SUCCESS]    A0A0I9Z3I8
[SUCCESS]    A0A0J6VW20
[SUCCESS]    A0A0J6VZP7
[SUCCESS]    A0A0J6WR15
[SUCCESS]    A0A0J6Z0M9
[SUCCESS]    A0A0J8TWL2  (from group of 2)
[SUCCESS]    A0A0J8TXD9
[SUCCESS]    A0A0K0X557
[SUCCESS]    A0A0K0XB28
[SUCCESS]    A0A0K0XCM7
[SUCCESS]    A0A0L0JLL3
[SUCCESS]    A0A0L0JTI1
[SUCCESS]    A0A0M2JYX2
[SUCCESS]    A0A0M2WG04
